# Week 4 — BANKING77 final test evaluation + analysis (Kunj)
Scores every locked checkpoint on the TEST set once, then builds the final tables, bootstrap intervals and figures.

**Before running:**
1. Right panel → **Add Input** → **Your Work** → **Notebooks** → add your **Week 2** notebook and your **Week 3** notebook. Their outputs hold the checkpoints.
2. Accelerator = **GPU T4 x2**, Internet = **On**.
3. **Save Version → Save & Run All (Commit)**. About 10–15 min.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one T4, matching the proposal
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/devjp99/lora-on-a-budget.git
%cd lora-on-a-budget
!git checkout kunj/week4-test-eval
!git log --oneline -3

In [ ]:
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10, which peft rejects; we don't use it
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ['PYTHONPATH'] = 'src'

## 1. Rebuild manifests and run tests

In [ ]:
!python scripts/make_banking77_manifest.py
!git status --short   # must print nothing: the eval script refuses to run on changed code
!LORA_BUDGET_ONLINE=1 python -m pytest -q

## 2. Find the checkpoints from the Week 2 and Week 3 outputs

In [ ]:
import glob
best = glob.glob('/kaggle/input/**/best/', recursive=True)
roots = sorted({os.path.dirname(os.path.dirname(p.rstrip('/'))) for p in best})
print(len(best), 'best checkpoints found (expect 13 or more)')
for r in roots:
    print(r, '->', len(os.listdir(r)), 'run folders')

## 3. Test-set evaluation (once)
For each run it first re-scores the tuning set and checks it matches the training log. Only then does it score the test set.

In [ ]:
!python scripts/eval_test_banking77.py --ckpt-roots {' '.join(roots)}

## 4. Analysis: tables, paired bootstrap, per-intent errors, figures

In [ ]:
!python scripts/analyze_banking77.py

In [ ]:
from IPython.display import Image, display
display(Image('results/analysis/fig_quality_cost.png'))
display(Image('results/analysis/fig_per_class_f1.png'))

## 5. Save results for download

In [ ]:
import shutil
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
shutil.make_archive(os.path.join(out_dir, 'week4_results'), 'zip', '.', 'results')
print('saved', os.path.join(out_dir, 'week4_results.zip'))